# 🌍 Day 3: AI-Powered GIS — APIs, Concepts, and Vibe Coding in Practice

## ITI · Gen AI Course · GIS Track

**Duration:** 6 hours (Instructor-led) + 6 hours (Self-study)

**Prerequisites:**
- Day 1 & 2 completed (Foundations + Vibe Coding)
- Basic Python familiarity (don't worry — the AI writes the code with you)
- A Google account (for free Gemini API)

---

## 🎯 Why is this lecture different?

Yesterday's vibe coding lecture taught you **how** to make the AI write code for you.

Today, the goal is **not** to write Python syntax by hand (that was the pre-ChatGPT world).

The goal is:

> **"Know what to ask the AI for, evaluate what it gives you back, and orchestrate AI APIs to solve real GIS problems."**

This is the difference between an engineer who uses AI well and one who copy-pastes and prays.

---

## 📋 Learning Objectives

By the end of today you will be able to:

1. **Distinguish Chat UI from API** and know when to use each
2. **Understand the anatomy** of any LLM call (system, user, tokens, temperature, streaming)
3. **Choose the right provider** (Gemini, Groq, OpenRouter, Local) based on the use case
4. **Use vibe coding** to build API integrations fast and with understanding
5. **Apply AI to real GIS use cases**: address parsing, satellite imagery analysis, QGIS scripting
6. **Know when AI fails** in GIS specifically (coordinate systems, hallucinated EPSG codes)

---

## 🛠️ Tools We'll Use

| Tool | Purpose | Free Tier |
|------|---------|-----------|
| **Google AI Studio (Gemini)** | Multimodal: text + images | 15 RPM, 1000 RPD |
| **Groq** | Fastest text inference | 30 RPM, 14.4K RPD |
| **OpenRouter** | Gateway to 100+ models | 20 RPM, 50 RPD |
| **Streamlit** | Quick web UIs | Free hosting |
| **Cursor / Claude Code** | Vibe coding (the IDE that writes the code) | Free tiers available |

---



---

## 🔧 Section 0: Setup — Universal API Key Helper

Before we touch anything, we need a helper function that loads API keys from any source.

This function works in:
- ✅ Google Colab (via Secrets)
- ✅ Local Jupyter / VS Code
- ✅ Any environment

**Run this cell first before anything else.**

> 💡 **Pedagogical note:** In the real world you would not write this code by hand. You would tell Cursor:
> *"Write a Python helper that loads API keys from Colab Secrets, environment variables, .env file, or asks the user."*
> And the AI writes it. Today we have it ready to save time.

In [1]:
# ============================================
# 🔑 UNIVERSAL API KEY HELPER
# Run this cell FIRST!
# ============================================

import os
from getpass import getpass

_API_KEYS = {}

def get_api_key(key_name: str, display_name: str = None) -> str:
    """Load API key from Colab Secrets, env vars, .env, or manual input."""
    if display_name is None:
        display_name = key_name

    # 1. Already loaded?
    if key_name in _API_KEYS and _API_KEYS[key_name]:
        return _API_KEYS[key_name]

    api_key = None
    source = None

    # 2. Try Colab Secrets
    try:
        from google.colab import userdata
        api_key = userdata.get(key_name)
        if api_key:
            source = "Colab Secrets"
    except Exception:
        pass

    # 3. Try environment variable
    if not api_key:
        api_key = os.environ.get(key_name)
        if api_key:
            source = "Environment Variable"

    # 4. Try .env file
    if not api_key:
        try:
            from dotenv import load_dotenv
            load_dotenv()
            api_key = os.environ.get(key_name)
            if api_key:
                source = ".env file"
        except ImportError:
            pass

    # 5. Manual input
    if not api_key:
        print(f"\n🔑 {display_name} API Key not found. Enter it now:")
        api_key = getpass(f"   {display_name} API Key: ")
        source = "Manual Input"

    _API_KEYS[key_name] = api_key
    print(f"✅ {display_name} loaded from: {source}")
    return api_key


def clear_api_keys():
    """Clear all stored API keys from memory."""
    global _API_KEYS
    _API_KEYS = {}
    print("🗑️ API keys cleared.")

print("✅ API Key Helper ready!")
print("📌 Usage: get_api_key('GOOGLE_API_KEY', 'Gemini')")

✅ API Key Helper ready!
📌 Usage: get_api_key('GOOGLE_API_KEY', 'Gemini')


---

# 📚 Part 1: Concepts (No Code Yet)

**Duration:** ~70 minutes

**Philosophy:** Before we write a single line, we have to understand **what's happening under the hood**. Later, when you use vibe coding, you'll be able to predict what the AI is going to write, and catch its mistakes quickly.

We'll cover three big questions:

1. **Why APIs at all?** (when we already have ChatGPT)
2. **What's the anatomy** of any LLM call?
3. **Which providers** exist and when to use which?

## 🤔 1.1 — Why APIs Matter (Even with Vibe Coding)

**Honest question:** If I have ChatGPT, why should I bother with an API?

The answer: **scale + automation + integration**.

### Practical comparison for a GIS engineer

| Scenario | Chat UI (ChatGPT) | API |
|----------|-------------------|-----|
| Geocode 5,000 addresses | ❌ Copy-paste 5,000 times? | ✅ One loop, done in minutes |
| Discuss a new project | ✅ Perfect fit | ❌ Overkill |
| Integrate into a QGIS plugin | ❌ Impossible | ✅ This is exactly what APIs are for |
| Write a regional report | ✅ Fast | ✅ Good if you need automation |
| Classify 10,000 satellite tiles | ❌ Impossible | ✅ Batch processing |
| Brainstorming | ✅ Best fit | ❌ Not the use case |

### The simple rule

> **Chat UI** = a conversation with the AI
> **API** = the AI as a component inside a larger workflow

### A real GIS example

Imagine you have a CSV with 500 addresses for urban development projects, and you want to:

1. Extract the district and governorate from each address
2. Geocode them to lat/lon
3. Create a 1 km buffer around each
4. Intersect with the land-use layer
5. Summarize the results in a report

Step (1) is where the LLM via API shines. The rest is traditional GIS tooling.

**The lesson:** AI is not a replacement for GIS — it's an **ingredient** in the pipeline.

---

### 💰 The cost angle

| | ChatGPT Plus | API (Gemini Flash) |
|---|--------------|---------------------|
| Monthly | $20 fixed | $0 within the free tier |
| 1 million tokens input | included | ~$0.075 |
| Scale | one person | 1000 users |
| Integration | manual | automated |

For personal use, ChatGPT is cheaper. For a production GIS app serving 100 users, the API is much cheaper.

## 🔬 1.2 — Anatomy of an LLM Call

Every API call to every LLM is built from the same building blocks. Learn them once and you understand every provider.

### The five components

```
┌──────────────────────────────────────────────────────────────┐
│                    ANATOMY OF AN LLM CALL                    │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  1. MESSAGES        →  The conversation (system + user + AI) │
│  2. MODEL           →  Which model (Gemini Flash, Llama...)  │
│  3. TEMPERATURE     →  Creativity level (0.0 → 2.0)          │
│  4. MAX_TOKENS      →  Maximum response length               │
│  5. STREAM          →  Whole response or token-by-token?     │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

---

### 🎭 Component 1: Messages (the conversation contract)

Every conversation with an LLM is made of three message types:

| Role | Who writes it? | What it does |
|------|---------------|--------------|
| `system` | You (the developer) | Sets the personality and the rules. Constant throughout the conversation. |
| `user` | The end user | The question or request |
| `assistant` | The AI | The reply. We keep it to maintain context. |

**GIS example:**

```
system:    "You are a senior GIS analyst specializing in remote sensing.
            Always cite EPSG codes when discussing projections."

user:      "What's the best CRS for mapping Egypt?"

assistant: "For Egypt, EGSA 1907 / Red Belt (EPSG:22992) is commonly used
            for cadastral work. For web mapping, Web Mercator (EPSG:3857)
            is standard..."

user:      "And for the Delta region specifically?"  ← AI must remember context
```

**Key insight:** The LLM is stateless by nature. With every request, you send the whole conversation. That's why long conversations cost more and degrade in quality.

---

### 🪙 Component 2: Tokens — the real currency

**What is a token?**
It's the unit of text the LLM actually sees. Not a character, not a word — a chunk in between.

**Rough rules:**
- 1 token ≈ 4 English characters
- 1 token ≈ ¾ of an English word
- Arabic uses more tokens (roughly 2–3× English)

**Examples:**

| Text | Tokens (approx.) |
|------|------------------|
| `Hello` | 1 |
| `Hello, how are you?` | 6 |
| `السلام عليكم` | ~8 |
| 500-word English response | ~375 |
| A 200-page book | ~80,000 |

**Why tokens matter:**

1. **Pricing** is per-token (not per-request)
2. **Context window** is measured in tokens (e.g., Gemini Flash = 1M tokens)
3. **Rate limits** are often TPM = Tokens Per Minute

**Practical tip:** When working with Arabic text, budget 2–3× the English cost.

---

### 🌡️ Component 3: Temperature

A parameter between 0 and 2 that controls how random the response is.

| Temperature | Behavior | Use it for |
|-------------|----------|------------|
| **0.0 — 0.2** | Deterministic, same answer every time | Code generation, structured extraction, classification |
| **0.3 — 0.7** | Balanced | Q&A, summarization, default chatbot |
| **0.8 — 1.2** | Creative | Brainstorming, creative writing |
| **1.3 — 2.0** | Wild | Experimentation — rarely useful |

**GIS examples:**

| Task | Temperature |
|------|-------------|
| Extract address components from messy text | 0.1 |
| Generate a QGIS Python script | 0.2 |
| Classify land use from satellite description | 0.1 |
| Suggest names for a new GIS project | 0.9 |
| Write a project narrative report | 0.5 |

**Heuristic:** If you want a "right vs wrong" answer → low temperature. If you want creativity → high temperature.

---

### 📏 Component 4: Max Tokens

The maximum length of the response.

**Why it matters:**
- **Cost control** — you don't want the AI generating 5000 tokens when you needed 100
- **Latency** — longer response = longer wait
- **Truncation** — set it too low and your response gets cut mid-sentence

**Rules of thumb:**
- Classification: 50 tokens
- Q&A: 500 tokens
- Long report: 2000–4000 tokens
- Code generation: 1000–2000 tokens

---

### 📡 Component 5: Streaming

**Non-streaming:** send request, wait 5–15 seconds, full response arrives.
**Streaming:** send request, tokens arrive one by one as the AI generates them.

```
Non-streaming:
[wait 8 seconds] ──→ "Here is the complete answer about Egypt's CRS..."

Streaming:
"Here" → "is" → "the" → "complete" → "answer" → ...
```

**When to use streaming:**
- ✅ Chatbots & UIs (the user sees text appearing → better UX)
- ✅ Long responses (feels interactive)

**When not to use streaming:**
- ❌ Background scripts (no benefit)
- ❌ Structured output (need the whole response to parse JSON)
- ❌ Batch processing (1000 requests with streaming = pain)

---

### 🪟 Bonus: Context Window

The maximum number of tokens the LLM can "see" at once (system prompt + entire conversation + response).

| Model | Context Window |
|-------|----------------|
| GPT-4o | 128K |
| Gemini 2.5 Flash | **1M** |
| Gemini 2.5 Pro | **2M** |
| Llama 3.3 70B | 128K |
| Claude Sonnet 4.6 | 200K |

**Why does this matter for GIS?**
If you want to fit 50 pages of spec documents plus a large attribute-table slice into the context, you need a model with a big context window. Gemini is the king here.

## 🏪 1.3 — The Provider Landscape

There are dozens of LLM providers. You only need to know four categories.

---

### Category 1: Closed Models (Premium)

| Provider | Best Model | Superpower | Free Tier? |
|----------|-----------|------------|------------|
| **OpenAI** | GPT-5 / GPT-4o | The most popular, mature ecosystem | ❌ (paid only) |
| **Anthropic** | Claude Opus / Sonnet | Best at reasoning and coding | ❌ (paid only) |
| **Google** | Gemini 2.5 Pro / Flash | **Multimodal + 1M context + free tier** | ✅ generous |

**Bottom line for GIS:** Start with **Gemini** — free, can see images (critical for satellite imagery), big context window.

---

### Category 2: Fast Inference (Open Models, Hosted)

| Provider | Hardware | Speed | Models |
|----------|----------|-------|--------|
| **Groq** | LPU (custom chip) | **~10× faster than competitors** | Llama, Mixtral, Qwen |
| **Together AI** | GPU clusters | Fast | 100+ open models |
| **Cerebras** | Wafer-scale chips | Fastest for text-only | Llama family |

**Bottom line for GIS:** For text-only chatbots needing instant responses, **Groq** wins.

---

### Category 3: Aggregators (Gateway to Many Models)

| Provider | Value proposition |
|----------|-------------------|
| **OpenRouter** | One API, 100+ models, auto-fallbacks, 28+ free models |
| **LiteLLM** | Library that converts any provider to OpenAI format |

**Bottom line for GIS:** If you want to experiment with many models without managing 10 API keys, **OpenRouter** solves it.

---

### Category 4: Local Models (Self-Hosted)

| Tool | Requirements | Use case |
|------|--------------|----------|
| **Ollama** | 8–16 GB RAM minimum | Development, prototyping |
| **LM Studio** | GUI for non-developers | Exploration |
| **vLLM** | GPU serving | Production with privacy needs |

**Why use local?**
- 🔒 **Privacy** — data never leaves your machine (critical for client data and defense work)
- 💵 **Cost** — no per-token billing
- 🌐 **Offline** — works without internet (useful for field surveys)

**Why it doesn't fit everything:**
- Needs hardware (a serious GPU is $2000+)
- Free open models are weaker than closed ones
- Maintenance and setup

---

### 🎯 Decision Tree: which to use for what?

```
Need to analyze an image (satellite, floor plan, map)?
    ├── YES → Gemini (only one with strong vision in the free tier)
    └── NO ↓

Is speed critical (real-time chatbot)?
    ├── YES → Groq (Llama 3.3 70B for quality, 8B for speed)
    └── NO ↓

Privacy critical (client data, NDA)?
    ├── YES → Ollama (local Llama or Gemma)
    └── NO ↓

Need to compare many models?
    ├── YES → OpenRouter
    └── NO → Gemini default (free + multimodal + strong)
```

---

### 🏆 Recommendation for the course

| Use Case | Provider | Model |
|----------|----------|-------|
| Default starter | Gemini | `gemini-2.5-flash` |
| Image analysis | Gemini | `gemini-2.5-flash` |
| Fast chatbot | Groq | `llama-3.3-70b-versatile` |
| Quick experiments | Groq | `llama-3.1-8b-instant` |
| Comparing models | OpenRouter | various `:free` |
| Production / Privacy | Ollama (local) | `gemma-4-27b` |

## ☕ 1.4 — Break + Q&A (15 minutes)

Before we drop into code, a few quick questions to confirm the concepts are clear:

1. If you want to classify 1000 satellite tiles as "urban / agricultural / desert", do you use the Chat UI or the API? Why?
2. If your system prompt says "You are a Python expert" and the user asks a GIS question, will the AI answer or refuse? Why?
3. If you run the same prompt twice with `temperature=0.0`, will the answer be identical?
4. Why is streaming not useful for background scripts?
5. You want to analyze a satellite image — do you use Groq Llama 70B or Gemini Flash? Why?

**(Answers are in Appendix A at the end of the notebook.)**

---

# 💻 Part 2: Live Vibe-Coding



**Philosophy:**

> *"You don't understand code by writing it. You understand it by reading it with intent."*

In this section, **you as the instructor** will build the code live using Cursor / Claude Code / GitHub Copilot. The students do **not** type. They **read**, **predict**, and **ask why**.

Every code cell is followed by **an explanation of what the AI produced** — so students understand what happened, not just `Shift+Enter`.

### 🎯 The demos:

| # | Demo | Skill |
|---|------|-------|
| 2.1 | First API call to Gemini | **Reading code with intent** |
| 2.2 | Iterating: streaming + system prompt + error handling | **Refactoring with AI** |
| 2.3 | Switching providers (Gemini → Groq → OpenRouter) | **Pattern recognition** |
| 2.4 | Multimodal: analyzing satellite imagery | **The GIS killer feature** |

## 🚀 2.1 — First API Call by Vibe Coding

**Time:** 30 minutes

### The scenario

I need a script that asks Gemini a simple GIS question and prints the answer.

### The prompt I'll give the AI:

```
Write Python code that:
1. Uses google-generativeai library to call Gemini 2.5 Flash
2. Asks 'What are the most common GIS file formats and their use cases?'
3. Prints the response

Use the get_api_key('GOOGLE_API_KEY', 'Gemini') helper that's already defined.
Keep the code minimal and well-commented.
```

### What the AI produces (after two seconds):

In [2]:
# Install the SDK (one-time setup)
!pip install -q google-generativeai

In [14]:
# ============================================
# DEMO 2.1: First Gemini API Call
# Generated by AI, read line-by-line below
# ============================================

import google.generativeai as genai

# Step 1: Load API key (using our helper from Section 0)
GOOGLE_API_KEY = get_api_key('GOOGLE_API_KEY', 'Gemini')

# Step 2: Configure the SDK with our key
genai.configure(api_key=GOOGLE_API_KEY)

# Step 3: Initialize the model
model = genai.GenerativeModel('gemini-2.5-flash')

# Step 4: Send a prompt and get a response
response = model.generate_content(
    "What are the most common GIS file formats and their use cases? "
    "Keep it concise — list 5 formats max."
)

# Step 5: Print the response
print(response.text)


🔑 Gemini API Key not found. Enter it now:


c:\Users\ibrah\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Gemini loaded from: Manual Input
Here are 5 common GIS file formats and their use cases:

1.  **Shapefile (.shp, .shx, .dbf, etc.)**
    *   **Use Cases:** The industry standard for storing **vector data** (points, lines, polygons). Widely used for sharing basic geographic features, creating maps, and general-purpose vector analysis due to its widespread compatibility.

2.  **GeoTIFF (.tif)**
    *   **Use Cases:** A standard for **raster data** that embeds georeferencing information directly into the TIFF image file. Commonly used for aerial photographs, satellite imagery, digital elevation models (DEMs), and other gridded datasets.

3.  **ESRI File Geodatabase (.gdb)**
    *   **Use Cases:** ESRI's modern, proprietary format for storing various GIS datasets (vector, raster, tables) in a single folder. Ideal for managing complex GIS projects, large datasets, and maintaining data integrity with features like topology and versioning within the ArcGIS ecosystem.

4.  **KML/KMZ (Keyhole

### 🔍 Read the code together (instructor walks through)

This is the important moment. Close the IDE where you were vibe-coding and open this notebook. Read with the students, line by line:

**`import google.generativeai as genai`**
We import Google's official SDK. The `as genai` is a rename to keep typing short.
**Ask the students:** Why not `import google.generativeai`? → Because typing `google.generativeai.configure()` every time is exhausting.

**`GOOGLE_API_KEY = get_api_key(...)`**
We use the helper from Section 0.
**Ask:** Why not just paste the key directly into the code (`api_key = 'AIza...'`)? → Security. If this code lands on GitHub, the key leaks.

**`genai.configure(api_key=...)`**
Global SDK configuration. Done once at the start of the script.

**`model = genai.GenerativeModel('gemini-2.5-flash')`**
We instantiate a model object. The string `'gemini-2.5-flash'` is the model ID.
**Ask:** Why `flash` and not `pro`? → Flash is cheaper and faster, and plenty for this task.

**`response = model.generate_content(...)`**
This is the actual API call. We send the prompt, the SDK talks to Google's servers, the response comes back.

**`response.text`**
The response object contains lots of data (tokens used, finish reason, safety ratings…). We only extract the text.

---

### ✨ The "Aha" Moment

**Tell the students:**

> *"You understand this code not because you wrote it. You understand it because you read it with intent. That's the fundamental difference between a competent vibe coder and one who copy-pastes."*

### 🧪 Try it yourself (5-minute exercise)

Change the question to something GIS-specific you're thinking about right now. For example:
- *"Explain the difference between WGS84 and Web Mercator in 3 sentences"*
- *"What's the best satellite imagery source for monitoring urban growth in Egypt?"*
- *"List 3 free alternatives to ArcGIS"*

## 🔄 2.2 — Iterating with AI: The Real Vibe Coding Skill

**Time:** 30 minutes

### The mindset

In the real world, the first version of code is rarely the final one. Every feature we add, we go back to the AI and say "add X".

We'll add 4 features to the previous code, one after another. Each time:
1. We'll see the prompt we send to the AI
2. We'll see the output
3. We'll understand the diff (what changed)

---

### Iteration 1: Add Streaming

**Prompt to the AI:**
```
Modify the code above to use streaming so we see the response as it's generated,
not all at once.
```

In [15]:
# ============================================
# ITERATION 1: Streaming
# ============================================

model = genai.GenerativeModel('gemini-2.5-flash')

# The key change: stream=True
response = model.generate_content(
    "What are the most common GIS file formats and their use cases? "
    "Keep it concise — list 5 formats max.",
    stream=True   # ← this is what changed
)

# Now we loop through chunks instead of getting the full response
print("Streaming response:")
for chunk in response:
    print(chunk.text, end="", flush=True)

print("\n\n✅ Stream complete!")

Streaming response:
Here are 5 of the most common GIS file formats and their use cases:

1.  **ESRI Shapefile (.shp, .shx, .dbf, etc.)**
    *   **Type:** Vector
    *   **Use Cases:** The de-facto industry standard for storing point, line, and polygon features. Widely used for general-purpose data exchange between different GIS software.

2.  **GeoTIFF (.tif, .tiff)**
    *   **Type:** Raster
    *   **Use Cases:** Stores georeferenced imagery and grid data, such as satellite images, aerial photographs, Digital Elevation Models (DEMs), and other raster datasets where each pixel has a geographical location.

3.  **ESRI File Geodatabase (.gdb)**
    *   **Type:** Vector & Raster Container
    *   **Use Cases:** ESRI's proprietary format for storing multiple GIS datasets (feature classes, raster datasets, tables) in a single folder. Excellent for managing large, complex projects and maintaining data integrity.

4.  **GeoJSON (.geojson)**
    *   **Type:** Vector
    *   **Use Cases:** A 

**What changed?**

1. We added `stream=True` to `generate_content()`
2. Instead of `response.text` (single value), we do `for chunk in response` (loop)
3. Each chunk contains a few tokens (not a single character)
4. `flush=True` makes the terminal print immediately instead of buffering

**Ask:** Why `end=""`? → Because `print` adds `\n` by default, and we don't want a newline after every chunk.

---

### Iteration 2: Add a System Prompt

We want the AI to behave as a specific GIS expert, not a generalist.

**Prompt to the AI:**
```
Add a system instruction that makes the model behave as a senior GIS analyst
who specializes in remote sensing for arid regions. Always mention EPSG codes
when discussing projections.
```

In [6]:
# ============================================
# ITERATION 2: System Prompt
# ============================================

# In Gemini, the system instruction is passed at model initialization
system_prompt = (
    "You are a senior GIS analyst with 15 years of experience in remote sensing "
    "for arid regions (Egypt, Saudi Arabia, UAE). "
    "When discussing projections, always cite the EPSG code. "
    "Be concise and technical — your audience is GIS engineers, not the general public."
)

model = genai.GenerativeModel(
    'gemini-2.5-flash',
    system_instruction=system_prompt   # ← the new piece
)

response = model.generate_content(
    "What's the best CRS for mapping the Nile Delta?",
    stream=True
)

for chunk in response:
    print(chunk.text, end="", flush=True)

For mapping the Nile Delta, the most appropriate CRS, ensuring minimal distortion and aligning with national mapping standards in Egypt, is:

1.  **Primary Recommendation (Local Datum):**
    *   **CRS:** Egyptian 1907 / Red Belt
    *   **EPSG:** `EPSG:22991`
    *   **Datum:** Egypt 1907
    *   **Projection:** Transverse Mercator (Central Meridian 31°E)
    *   **Rationale:** This system is specifically designed for Egypt's main inhabited areas, including the Delta, providing excellent accuracy and low distortion over its extent. It is based on a local geodetic datum.

2.  **Alternative (Global Datum - WGS84):**
    If WGS84 compatibility is paramount for your source data or interoperability, you would need to consider the relevant UTM zones. The Nile Delta spans two UTM zones:
    *   **Western Delta:** WGS 84 / UTM Zone 35N (`EPSG:32635`)
    *   **Eastern Delta:** WGS 84 / UTM Zone 36N (`EPSG:32636`)
    *   **Rationale:** While globally recognized, using a single UTM zone for th

**Notice the changes:**

1. We defined `system_prompt` as a separate string — the best practice
2. We pass it to `GenerativeModel` as `system_instruction`
3. The user question is now short (one line) — because the context is in the system prompt

**Try it:** Change the system prompt to `"You are a GIS instructor explaining to first-year students"` and see the difference on the same question.

---

### Iteration 3: Add Error Handling

In the real world, the API can:
- Drop the connection (network error)
- Return a rate-limit error
- Return an empty response
- Reject for safety reasons

**Prompt to the AI:**
```
Wrap the API call in proper error handling with try/except.
Handle network errors, rate limits, and empty responses gracefully.
```

In [7]:
# ============================================
# ITERATION 3: Error Handling
# ============================================

import google.generativeai as genai
from google.api_core import exceptions as google_exceptions

def ask_gemini(question: str, system_prompt: str = None) -> str:
    """Ask Gemini a question, with proper error handling."""
    try:
        model = genai.GenerativeModel(
            'gemini-2.5-flash',
            system_instruction=system_prompt
        )

        response = model.generate_content(question)

        # Check if response was blocked or empty
        if not response.text:
            return "⚠️ Empty response (possibly blocked by safety filters)"

        return response.text

    except google_exceptions.ResourceExhausted:
        return "⚠️ Rate limit hit. Wait a bit and try again."

    except google_exceptions.PermissionDenied:
        return "⚠️ API key invalid or quota exceeded."

    except google_exceptions.GoogleAPIError as e:
        return f"⚠️ Google API error: {e}"

    except Exception as e:
        return f"⚠️ Unexpected error: {type(e).__name__}: {e}"

# Test it
answer = ask_gemini(
    "What's the EPSG code for Egypt's national grid?",
    system_prompt="You are a GIS expert. Be brief."
)
print(answer)

EPSG:5229 (ETRS89 / Egypt National Grid 2007)


**What we learned:**

1. We wrapped the code in a `function` — reusable
2. Specific exceptions are better than `except Exception:` — we know exactly what failed
3. `ResourceExhausted` is the rate-limit case — must be handled in production
4. Even when the API returns a response, it can be empty (safety filters)

**Important point:** The AI wrote error handling better than 80% of junior developers do. That's one of the gifts of vibe coding — you get best practices for free.

---

### Iteration 4: Refactor for Reusability

**Prompt to the AI:**
```
Add support for multi-turn conversations (chat history).
The function should accept a list of past messages and continue the conversation.
```

In [8]:
# ============================================
# ITERATION 4: Multi-turn Conversation
# ============================================

def chat_with_gemini(
    messages: list,
    system_prompt: str = None,
    model_name: str = 'gemini-2.5-flash'
) -> str:
    """
    Multi-turn conversation with Gemini.

    messages format:
        [
            {'role': 'user', 'content': '...'},
            {'role': 'assistant', 'content': '...'},
            {'role': 'user', 'content': '...'}
        ]
    """
    try:
        model = genai.GenerativeModel(model_name, system_instruction=system_prompt)

        # Gemini uses 'model' instead of 'assistant', and 'parts' instead of 'content'
        gemini_history = []
        for msg in messages[:-1]:  # all but last
            role = 'model' if msg['role'] == 'assistant' else 'user'
            gemini_history.append({'role': role, 'parts': [msg['content']]})

        chat = model.start_chat(history=gemini_history)
        response = chat.send_message(messages[-1]['content'])

        return response.text

    except Exception as e:
        return f"⚠️ Error: {e}"

# Test multi-turn conversation
conversation = [
    {'role': 'user', 'content': 'What is the EPSG code for WGS84?'},
    {'role': 'assistant', 'content': 'EPSG:4326 represents WGS84 geographic CRS.'},
    {'role': 'user', 'content': 'And for Web Mercator?'},  # ← this depends on context
]

system = "You are a GIS expert. Always cite EPSG codes. Be concise."
answer = chat_with_gemini(conversation, system_prompt=system)
print(answer)

EPSG:3857 is the standard code for Web Mercator (also known as WGS 84 / Pseudo-Mercator). EPSG:900913 is an older, non-official code that refers to the same projection.


**Pedagogical notes on the refactor:**

1. **Gemini quirk:** it uses `'model'` instead of `'assistant'`, and `'parts'` instead of `'content'`. This differs from the OpenAI format. The AI handled the mapping automatically.

2. **`start_chat(history=...)`:** Gemini has a chat-session concept that maintains context. We put every message except the last into history, and send the last one via `send_message`.

3. **Type hints:** `messages: list, system_prompt: str = None`. They don't enforce types, but they make the code's intent clear.

**Ask the students:**
In the last question `"And for Web Mercator?"`, why did the AI understand that the question is about EPSG codes?
→ Because the `history` carries the context. Without it the AI would have answered "Web Mercator is a projection used for..."

## 🔀 2.3 — Switching Providers (Same Logic, Different Backend)

**Time:** 30 minutes

### The big insight

Every modern LLM API follows roughly **the same pattern**:

```
1. Initialize a client with an API key
2. Build a messages list (system + user + assistant)
3. Call create() with model name and parameters
4. Read response.choices[0].message.content
```

This is called **OpenAI-compatible format**, and most providers follow it. Gemini is a bit different but offers a compatibility mode.

### The demo: same question, 3 different providers

We'll ask the same question (`"What's a Shapefile?"`) from 3 providers and compare:
- Speed
- Answer quality
- Code shape (we'll see the pattern)

In [2]:
# Install all the SDKs we need
!pip install --upgrade pip
!pip install -q groq openai

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 2.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.7 MB/s  0:00:00



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\ibrah\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\ibrah\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\ibrah\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


---

### Provider 1: Groq (Llama 3.3 70B)

**Prompt to the AI:**
```
Write a function ask_groq(question) that calls Groq's Llama 3.3 70B model
and returns the response. Use the get_api_key helper.
```

In [6]:
# ============================================
# PROVIDER 1: GROQ
# ============================================

%pip install -q groq

from groq import Groq
import time

def ask_groq(question: str, system: str = None) -> tuple[str, float]: 
    """Ask Groq Llama 3.3 70B. Returns (answer, elapsed_seconds)."""
    groq_client = Groq(api_key=get_api_key('GROQ_API_KEY', 'Groq'))

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})

    start = time.time()
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=0.3,
        max_tokens=300,
    )
    elapsed = time.time() - start

    return response.choices[0].message.content, elapsed

# Test it
answer, elapsed = ask_groq(
    "What's a Shapefile? Answer in 3 sentences.",
    system="You are a GIS expert. Be precise and technical."
)
print(f"⚡ Time: {elapsed:.2f}s\n")
print(answer)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


⚡ Time: 0.64s

A Shapefile is a geospatial data format used to store geometric data, such as points, lines, and polygons, along with their associated attributes. It consists of a main file with a .shp extension, accompanied by auxiliary files with .shx and .dbf extensions, which together contain the geometric data, spatial index, and attribute data. Developed by Esri, Shapefiles are widely supported by geographic information systems (GIS) software, including ArcGIS, QGIS, and others, making them a popular choice for exchanging and storing geospatial data.


**Notice:**

1. **`Groq(api_key=...)`** — the SDK is instantiated with the API key. No global `configure()` like Gemini.
2. **`messages` list** — system and user are dicts. This is the OpenAI format.
3. **`groq_client.chat.completions.create(...)`** — the API call. The signature is identical to OpenAI's.
4. **`response.choices[0].message.content`** — the same path to extract the text.
5. The request usually takes < 1 second. Groq is very fast.

**Ask:** Why `choices[0]`? → Technically the API can return multiple alternative responses (parameter `n`). We request one, so we take index 0.

---

### Provider 2: OpenRouter (Gateway to Many Models)

OpenRouter uses the OpenAI SDK as-is, just with a different base URL.

**Prompt to the AI:**
```
Write the same function but for OpenRouter, using a free model.
Use the OpenAI SDK with OpenRouter's base URL.
```

In [3]:
# ============================================
# PROVIDER 2: OPENROUTER
# ============================================

from openai import OpenAI

def ask_openrouter(question: str, system: str = None,
                   model: str = "openrouter/free") -> tuple[str, float]:
    """Ask any OpenRouter model. Returns (answer, elapsed_seconds)."""
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",   # ← the trick
        api_key=get_api_key('OPENROUTER_API_KEY', 'OpenRouter'),
    )

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})

    start = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.3,
        max_tokens=300,
    )
    elapsed = time.time() - start

    return response.choices[0].message.content, elapsed

# Test it
answer, elapsed = ask_openrouter(
    "What's a Shapefile? Answer in 3 sentences.",
    system="You are a GIS expert. Be precise and technical."
)
print(f"⚡ Time: {elapsed:.2f}s\n")
print(answer)


🔑 OpenRouter API Key not found. Enter it now:
✅ OpenRouter loaded from: Manual Input


NameError: name 'time' is not defined

**The magic:**

Look at the difference between `ask_groq` and `ask_openrouter`. **The body is ~100% identical.** The only differences:

| Groq | OpenRouter |
|------|------------|
| `from groq import Groq` | `from openai import OpenAI` |
| `Groq(api_key=...)` | `OpenAI(base_url=..., api_key=...)` |
| `model="llama-3.3-70b-versatile"` | `model="meta-llama/llama-3.3-70b-instruct:free"` |

**This is what we call OpenAI-compatible format.** Learn one provider, you've learned them all.

**Ask:** What does `:free` mean at the end of the model name on OpenRouter? → It's a suffix indicating the free version (lower rate limits but no charges).

---

### Provider 3: Gemini (for comparison)

Gemini has its own SDK but the same concepts.

In [22]:
# ============================================
# PROVIDER 3: GEMINI (reusing our function from 2.2)
# ============================================

def ask_gemini_timed(question: str, system: str = None) -> tuple[str, float]:
    """Ask Gemini. Returns (answer, elapsed_seconds)."""
    model = genai.GenerativeModel(
        'gemini-2.5-flash',
        system_instruction=system
    )

    start = time.time()
    response = model.generate_content(question)
    elapsed = time.time() - start

    return response.text, elapsed

# Test it
answer, elapsed = ask_gemini_timed(
    "What's a Shapefile? Answer in 3 sentences.",
    system="You are a GIS expert. Be precise and technical."
)
print(f"⚡ Time: {elapsed:.2f}s\n")
print(answer)

⚡ Time: 2.83s

A Shapefile is a digital vector data storage format developed by Esri for storing geographic features such as points, lines, and polygons, along with their associated attribute data. It is not a single file, but rather a collection of files with mandatory extensions including `.shp` (main file containing geometry), `.shx` (index file), and `.dbf` (dBASE table for attributes), often accompanied by others like `.prj` for projection. Despite its age and lack of support for complex topology or 3D surfaces, the Shapefile remains a widely adopted and open specification for geospatial data exchange across various GIS software.


---

### 🏁 Head-to-Head Comparison

Let's ask the same question from all three and compare:

In [23]:
# ============================================
# COMPARE ALL THREE PROVIDERS
# ============================================

question = "What's the best CRS for mapping coastal erosion in the Nile Delta? Be brief."
system = "You are a senior remote sensing analyst. Cite EPSG codes."

providers = {
    "🔵 Gemini 2.5 Flash":     ask_gemini_timed,
    "⚡ Groq Llama 3.3 70B":   ask_groq,
    "🌐 OpenRouter (free)":    ask_openrouter,
}

results = {}
for name, func in providers.items():
    print(f"\n{'='*60}")
    print(f"{name}")
    print('='*60)
    try:
        answer, elapsed = func(question, system=system)
        print(f"⏱️  {elapsed:.2f}s\n")
        print(answer)
        results[name] = elapsed
    except Exception as e:
        print(f"❌ Error: {e}")

print(f"\n{'='*60}")
print("⚡ SPEED RANKING")
print('='*60)
for name, elapsed in sorted(results.items(), key=lambda x: x[1]):
    print(f"  {name:30s} → {elapsed:.2f}s")


🔵 Gemini 2.5 Flash
⏱️  6.68s

For mapping coastal erosion in the Nile Delta, **WGS 84 / UTM zone 36N (EPSG:32636)** is ideal.

It's a projected CRS that minimizes distortion for accurate area and distance measurements across the region. The underlying geographic CRS is **WGS 84 (EPSG:4326)**.

⚡ Groq Llama 3.3 70B

🔑 Groq API Key not found. Enter it now:
✅ Groq loaded from: Manual Input
❌ Error: Connection error.

🌐 OpenRouter (free)

🔑 OpenRouter API Key not found. Enter it now:
✅ OpenRouter loaded from: Manual Input
❌ Error: Error code: 401 - {'error': {'message': 'No cookie auth credentials found', 'code': 401}}

⚡ SPEED RANKING
  🔵 Gemini 2.5 Flash             → 6.68s


### 📊 Discussion (with the students)

After running the comparison, ask the students:

1. **Who was fastest?** Usually Groq. Why? → LPU hardware instead of GPU.
2. **Whose answer was deepest?** Usually Gemini or Groq Llama 70B.
3. **If you build a production chatbot, which provider?** → Depends:
   - Simple customer-support chatbot → Groq (speed + cost)
   - Chatbot that analyzes reports or images → Gemini (multimodal + big context)
   - Exploration and A/B testing → OpenRouter

### 🧠 The Hidden Lesson

> When you write a function that abstracts the provider, you're buying **insurance** against lock-in.
> If Gemini raises prices tomorrow, you can switch to Groq with a single line change.
> That's software engineering in the AI era.

## 🛰️ 2.4 — Multimodal: The GIS Killer Feature

**Time:** 45 minutes

### Why is this the most important part of the lecture for you?

As GIS engineers, your daily work revolves around:
- 🗺️ Maps
- 🛰️ Satellite imagery
- 📊 Charts and plots
- 📸 Field photos

All of these are **images**. And as of 2025+, LLMs can **see** and analyze these images. This is a fundamental shift in how you work.

### Who supports vision?

| Provider | Vision Support | Quality |
|----------|----------------|---------|
| **Gemini 2.5 Flash** | ✅ Native, in the free tier | Excellent |
| **Gemini 2.5 Pro** | ✅ Native | The best |
| **GPT-4o** | ✅ | Excellent (paid only) |
| **Claude Sonnet/Opus** | ✅ | Excellent (paid only) |
| **Groq Llama 4 Scout** | ✅ vision-enabled | Good, free |
| **DeepSeek** | ❌ Text only | — |

**Bottom line:** For GIS work, **Gemini is the default**. Free, works, fast.

### The 4 GIS use cases for multimodal

1. **Satellite image analysis** — "identify land use types"
2. **Map reading** — "what does this contour map tell us?"
3. **Field photo classification** — "is this concrete or asphalt?"
4. **Chart/plot understanding** — "summarize this NDVI time series"

---

### Demo 1: Satellite Image Analysis

**The scenario:** we have a satellite image and we want the AI to identify:
- The land-use types present
- Approximate percentages of each
- Urban planning notes

**Prompt to the AI:**
```
Write code that downloads a sample satellite image (use any public one)
and asks Gemini to identify land use types in it.
```

In [25]:
# ============================================
# MULTIMODAL DEMO 1: Satellite Image Analysis
# ============================================

import PIL.Image
import requests
from io import BytesIO

# Public satellite image (USGS / NASA imagery is free to use)
# This is a Landsat image of the Nile Delta
image_url = (
    "https://eoimages.gsfc.nasa.gov/images/imagerecords/89000/89656/"
    "niledelta_oli_2017006_lrg.jpg"
)

# Fallback image if the primary URL is unavailable
fallback_image_url = (
    "https://via.placeholder.com/1200x800.png?text=Nile+Delta+Satellite+Image"
)

def download_image(url: str) -> PIL.Image.Image:
    response = requests.get(url)
    response.raise_for_status()
    return PIL.Image.open(BytesIO(response.content))

# Download the image
try:
    img = download_image(image_url)
except Exception as exc:
    print(f"⚠️ Failed to download primary image ({exc}). Using fallback image.")
    img = download_image(fallback_image_url)

# Display it inline
try:
    from IPython.display import display
    display(img.resize((600, 600)))
except Exception:
    print(f"Image loaded: {img.size}")

print(f"\n📐 Original size: {img.size}")
print(f"📦 Mode: {img.mode}")

⚠️ Failed to download primary image (404 Client Error: Not Found for url: https://eoimages.gsfc.nasa.gov/images/imagerecords/89000/89656/niledelta_oli_2017006_lrg.jpg). Using fallback image.


SSLError: HTTPSConnectionPool(host='via.placeholder.com', port=443): Max retries exceeded with url: /1200x800.png?text=Nile+Delta+Satellite+Image (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:992)')))

In [27]:
# ============================================
# Send the image to Gemini with a GIS-focused prompt
# ============================================

import PIL.Image
import requests
from io import BytesIO

# Public satellite image (USGS / NASA imagery is free to use)
# This is a Landsat image of the Nile Delta
image_url = (
    "https://eoimages.gsfc.nasa.gov/images/imagerecords/89000/89656/"
    "niledelta_oli_2017006_lrg.jpg"
)

# Fallback image if the primary URL is unavailable
fallback_image_url = (
    "https://via.placeholder.com/1200x800.png?text=Nile+Delta+Satellite+Image"
)

def download_image(url: str) -> PIL.Image.Image:
    response = requests.get(url)
    response.raise_for_status()
    return PIL.Image.open(BytesIO(response.content))

# Download the image
try:
    img = download_image(image_url)
except Exception as exc:
    print(f"⚠️ Failed to download primary image ({exc}). Using fallback image.")
    img = download_image(fallback_image_url)

system_prompt = (
    "You are a remote sensing analyst. Analyze satellite imagery for:\n"
    "1. Land use / land cover classification\n"
    "2. Approximate area percentages\n"
    "3. Notable features (rivers, urban areas, agriculture)\n"
    "Be concise and use technical vocabulary."
)

model = genai.GenerativeModel(
    'gemini-2.5-flash',
    system_instruction=system_prompt
)

# The magic: pass image + text together
response = model.generate_content([
    "Analyze this satellite image. Identify land cover types, estimate percentages, "
    "and provide any insights relevant to urban or agricultural planning.",
    img   # ← the image object as a regular input
])

print(response.text)

⚠️ Failed to download primary image (404 Client Error: Not Found for url: https://eoimages.gsfc.nasa.gov/images/imagerecords/89000/89656/niledelta_oli_2017006_lrg.jpg). Using fallback image.


SSLError: HTTPSConnectionPool(host='via.placeholder.com', port=443): Max retries exceeded with url: /1200x800.png?text=Nile+Delta+Satellite+Image (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:992)')))

**What happened under the hood:**

1. The `PIL.Image` object was serialized as base64 and sent in the request
2. The Gemini vision model saw the image and analyzed it as pixels
3. The response describes the region based on its visual features

**Notice:** the syntax is dead simple — a list with a string and an image. Gemini handles the rest.

**Warning:** the AI is **estimating**, not classifying at pixel-level accuracy. For serious land cover classification use traditional ML (e.g. Random Forest on Sentinel-2 bands). LLMs are great for **quick exploration and reporting**.

---

### Demo 2: Structured Output From an Image

Instead of a free-form paragraph, let's force the AI to return JSON so we can process the result in a pipeline.

In [29]:
# ============================================
# MULTIMODAL DEMO 2: Structured JSON Output
# ============================================

import json
import re
import requests
from io import BytesIO

structured_prompt = (
    "Analyze the satellite image and return a JSON object with this exact structure:\n"
    "{\n"
    '  "land_cover": [\n'
    '    {"type": "...", "percentage": 0-100, "description": "..."},\n'
    "    ...\n"
    "  ],\n"
    '  "dominant_feature": "...",\n'
    '  "region_hypothesis": "...",\n'
    '  "confidence": "low|medium|high"\n'
    "}\n"
    "Return ONLY the JSON, no markdown fences, no explanation."
)

model = genai.GenerativeModel('gemini-2.5-flash')

# ============================================
# MULTIMODAL DEMO 2: Structured JSON Output
# ============================================

import PIL.Image

# Public satellite image (USGS / NASA imagery is free to use)
# This is a Landsat image of the Nile Delta
image_url = (
    "https://eoimages.gsfc.nasa.gov/images/imagerecords/89000/89656/"
    "niledelta_oli_2017006_lrg.jpg"
)

# Fallback image if the primary URL is unavailable
fallback_image_url = (
    "https://via.placeholder.com/1200x800.png?text=Nile+Delta+Satellite+Image"
)

def download_image(url: str) -> PIL.Image.Image:
    response = requests.get(url)
    response.raise_for_status()
    return PIL.Image.open(BytesIO(response.content))

# Download the image
try:
    img = download_image(image_url)
except Exception as exc:
    print(f"⚠️ Failed to download primary image ({exc}). Using fallback image.")
    img = download_image(fallback_image_url)

structured_prompt = (
    "Analyze the satellite image and return a JSON object with this exact structure:\n"
    "{\n"
    '  "land_cover": [\n'
    '    {"type": "...", "percentage": 0-100, "description": "..."},\n'
    "    ...\n"
    "  ],\n"
    '  "dominant_feature": "...",\n'
    '  "region_hypothesis": "...",\n'
    '  "confidence": "low|medium|high"\n'
    "}\n"
    "Return ONLY the JSON, no markdown fences, no explanation."
)

model = genai.GenerativeModel('gemini-2.5-flash')

response = model.generate_content(
    [structured_prompt, img],
    generation_config={
        'temperature': 0.1,    # Low temp for structured output
        'response_mime_type': 'application/json',   # Gemini supports JSON mode
    }
)

raw = response.text

# Parse the JSON
try:
    data = json.loads(raw)
    print("✅ Parsed JSON successfully\n")
    print(json.dumps(data, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    # Fallback: strip markdown fences if AI ignored instructions
    cleaned = re.sub(r'^```(?:json)?|```$', '', raw.strip(), flags=re.MULTILINE).strip()
    data = json.loads(cleaned)
    print("✅ Parsed after cleanup\n")
    print(json.dumps(data, indent=2, ensure_ascii=False))

raw = response.text

# Parse the JSON
try:
    data = json.loads(raw)
    print("✅ Parsed JSON successfully\n")
    print(json.dumps(data, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    # Fallback: strip markdown fences if AI ignored instructions
    cleaned = re.sub(r'^```(?:json)?|```$', '', raw.strip(), flags=re.MULTILINE).strip()
    data = json.loads(cleaned)
    print("✅ Parsed after cleanup\n")
    print(json.dumps(data, indent=2, ensure_ascii=False))

⚠️ Failed to download primary image (404 Client Error: Not Found for url: https://eoimages.gsfc.nasa.gov/images/imagerecords/89000/89656/niledelta_oli_2017006_lrg.jpg). Using fallback image.


SSLError: HTTPSConnectionPool(host='via.placeholder.com', port=443): Max retries exceeded with url: /1200x800.png?text=Nile+Delta+Satellite+Image (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:992)')))

**This is the game-changer:**

Now you have `data` as a Python dict. You can:
- Save it to PostgreSQL
- Convert it to GeoJSON
- Join it with a shapefile
- Visualize it in QGIS

**New concept: `response_mime_type='application/json'`**
That's **JSON mode** in Gemini. It guarantees the output is valid JSON, not free text.
Groq and OpenAI have the same feature under different names (`response_format`).

---

### Demo 3: Multiple Images Comparison

Scenario: I have two images of the same area at different times. I want the AI to identify the changes.

In [30]:
# ============================================
# MULTIMODAL DEMO 3: Change Detection (Conceptual)
# ============================================

# For demo, we use the same image twice — replace with a real before/after pair
# (e.g., Sentinel-2 imagery from 2020 vs 2025 for the same area)
img_before = img  # placeholder
img_after  = img  # placeholder

change_prompt = (
    "You are looking at two satellite images of the same region taken at "
    "different times. The first is 'before', the second is 'after'.\n\n"
    "Identify any changes:\n"
    "- New urban development\n"
    "- Vegetation changes\n"
    "- Water body changes\n"
    "- Any anomalies\n\n"
    "If there are no changes (e.g., same image), say so clearly."
)

model = genai.GenerativeModel('gemini-2.5-flash')

response = model.generate_content([
    change_prompt,
    "BEFORE image:", img_before,
    "AFTER image:",  img_after,
])

print(response.text)

NameError: name 'img' is not defined

**Practical note for the real world:**

For real change detection, the correct workflow is:
1. **Co-register** the two images (same pixels in the same place)
2. Run **NDVI difference** or **change vector analysis**
3. Feed the LLM the **results** (not the raw images) so it can write the report

LLMs are weak at pixel-level precision, strong at narrative.

---

### 🧠 The Vibe Coding Reflection

In the last 45 minutes you saw:
- 3 multimodal demos
- Structured JSON output
- A change-detection concept

**All of this code was written by the AI.** I, the instructor, explained what's happening and why.

**The skill you acquired:**
- Reading code with understanding
- Knowing the capabilities (multimodal, JSON mode, conversation history)
- Knowing the limitations (LLMs are not a replacement for traditional ML for serious classification)

In Part 3, we'll apply these skills to 3 real GIS use cases.

---

# 🌍 Part 3: GIS Applications (Real Use Cases)

**Duration:** ~90 minutes

### The mindset

In Part 2 we saw **how** to talk to the API.
In Part 3 we'll see **why** and **when** to use it in real GIS work.

### 3 deliberately different use cases

| # | Use Case | Pattern |
|---|----------|---------|
| 3.1 | Smart Address Parser | **Structured Extraction** — turning messy text into organized data |
| 3.2 | Satellite Image Q&A | **Conversational Vision** — a chatbot that answers questions about images |
| 3.3 | QGIS Script Generator | **Code Generation** — the AI writes GIS scripts |
| 3.4 | When AI Fails | **Failure modes** — when not to trust the AI |

## 📍 3.1 — Use Case 1: Smart Address Parser

**Time:** 30 minutes

### The real problem

You have a CSV with 500 Egyptian addresses from a field survey. The format is chaotic:

```
شارع 9 المعادي القاهرة
المعادي - شارع تسعة - الدور الثالث
Maadi, Cairo, St 9
9 ش المعادي - بجوار مسجد النور
12 شارع جامعة الدول العربية، المهندسين، الجيزة
```

You need to extract:
- Street name
- District
- Governorate
- House number (if present)

### The traditional approach: Regex

```python
# You'd write 100 regex patterns and every time hit a new edge case that breaks them
pattern = r'(\d+)?\s*(ش|شارع|st|street)\s+(.+?)\s*[,،\-]?\s*(\w+)\s*[,،\-]?\s*(\w+)'
# 🔥 hell
```

**Regex fails because:**
- Languages are mixed (Arabic + English)
- Order is not consistent
- There are many abbreviations (`ش` = `شارع`)
- There are landmarks ("next to mosque")

### The AI approach: Structured extraction

We make the LLM understand the semantic content and return structured data.

In [31]:
# ============================================
# 3.1 — Smart Address Parser
# ============================================

import json
from typing import Optional

PARSER_SYSTEM_PROMPT = """You are an expert in Egyptian addresses (Arabic and English).
Your task is to extract structured components from messy address strings.

Rules:
1. Handle both Arabic and English text (mixed is common)
2. Normalize: "ش" → "شارع", "st" → "street"
3. Translate place names to English in the output
4. If a field is missing, use null
5. Confidence: "high" (clear), "medium" (some ambiguity), "low" (mostly guessing)

Output ONLY valid JSON. No markdown, no explanation.
"""

JSON_SCHEMA = """
{
  "street_name": "string or null",
  "street_number": "string or null",
  "district": "string or null",
  "governorate": "string or null",
  "landmark": "string or null",
  "original_language": "arabic|english|mixed",
  "confidence": "high|medium|low"
}
"""

def parse_address(address: str) -> dict:
    """Parse a messy address string into structured components."""
    model = genai.GenerativeModel(
        'gemini-2.5-flash',
        system_instruction=PARSER_SYSTEM_PROMPT
    )

    prompt = f"Address: {address}\n\nReturn JSON with this schema:\n{JSON_SCHEMA}"

    response = model.generate_content(
        prompt,
        generation_config={
            'temperature': 0.1,
            'response_mime_type': 'application/json',
        }
    )

    return json.loads(response.text)

# Test on the messy examples
messy_addresses = [
    "شارع 9 المعادي القاهرة",
    "المعادي - شارع تسعة - الدور الثالث",
    "Maadi, Cairo, St 9",
    "9 ش المعادي - بجوار مسجد النور",
    "12 شارع جامعة الدول العربية، المهندسين، الجيزة",
]

for addr in messy_addresses:
    print(f"\n📍 Input: {addr}")
    print("-" * 60)
    try:
        result = parse_address(addr)
        print(json.dumps(result, indent=2, ensure_ascii=False))
    except Exception as e:
        print(f"❌ Error: {e}")


📍 Input: شارع 9 المعادي القاهرة
------------------------------------------------------------
{
  "street_name": "Street 9",
  "street_number": null,
  "district": "Maadi",
  "governorate": "Cairo",
  "landmark": null,
  "original_language": "arabic",
  "confidence": "high"
}

📍 Input: المعادي - شارع تسعة - الدور الثالث
------------------------------------------------------------
{
  "street_name": "Street 9",
  "street_number": null,
  "district": "Maadi",
  "governorate": null,
  "landmark": "Third Floor",
  "original_language": "arabic",
  "confidence": "high"
}

📍 Input: Maadi, Cairo, St 9
------------------------------------------------------------
{
  "street_name": "Street 9",
  "street_number": null,
  "district": "Maadi",
  "governorate": "Cairo",
  "landmark": null,
  "original_language": "english",
  "confidence": "high"
}

📍 Input: 9 ش المعادي - بجوار مسجد النور
------------------------------------------------------------
{
  "street_name": "Al Maadi Street",
  "street_numb

### 📊 The output

Every address is now a structured Python dict. You can now:

1. **Build a DataFrame** and export a clean CSV
2. **Filter** by district or governorate
3. **Pass to a geocoding service** (Nominatim, Google Maps API)
4. **Join with a shapefile** of governorates

### 💡 Production tips

1. **Batch processing:** instead of 500 API calls, put 20 addresses in one prompt and have the AI return an array. Save on rate limits.

2. **Validation:** add post-processing — if the governorate isn't in a known list (Cairo, Giza, …), set it to `null`.

3. **Caching:** if addresses are duplicated, cache the result in a dict. The LLM shouldn't be queried twice for the same input.

4. **Confidence scoring:** the confidence field is very useful — you can flag the low ones for manual review.

### ⚠️ Limitations

- ❌ The LLM does **not** geocode (it doesn't return lat/lon). You need another geocoding service (Nominatim/Google).
- ❌ It can "invent" districts (hallucination). Always validate.
- ❌ For sensitive data (medical, financial), consider a local model instead of a cloud API.

## 🛰️ 3.2 — Use Case 2: Satellite Image Q&A App

**Time:** 30 minutes

### The scenario

We'll build a **Streamlit web app** that lets the user:
1. Upload a satellite image (or enter a URL)
2. Ask any question about it
3. See the response stream in

This is the pattern for "chat with your data" applications.

### Vibe-coding prompt

```
Create a Streamlit app called `satellite_chat.py` that:
1. Has a sidebar with API key input and model selection
2. Lets the user upload an image (PNG/JPG)
3. Displays the uploaded image
4. Shows a chat interface to ask questions about the image
5. Uses Gemini 2.5 Flash with streaming
6. Maintains conversation history with the image as context
```

### What the AI produces:

In [32]:
# ============================================
# 3.2 — Satellite Image Q&A Streamlit App
# Generated via vibe coding, then explained
# ============================================

streamlit_app_code = '''import streamlit as st
import google.generativeai as genai
from PIL import Image
import os

# ─────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────
st.set_page_config(
    page_title="🛰️ Satellite Image Chat",
    page_icon="🛰️",
    layout="wide"
)

st.title("🛰️ Satellite Image Q&A")
st.caption("Powered by Gemini 2.5 Flash · Built for GIS engineers")

# ─────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Settings")

    api_key = st.text_input(
        "Gemini API Key",
        type="password",
        value=os.environ.get("GOOGLE_API_KEY", ""),
        help="Get one free at aistudio.google.com"
    )

    model_choice = st.selectbox(
        "Model",
        ["gemini-2.5-flash", "gemini-2.5-pro"],
        help="Flash = faster + cheaper. Pro = smarter."
    )

    system_prompt = st.text_area(
        "System Prompt",
        value=("You are a remote sensing analyst. "
               "Analyze imagery with technical precision. "
               "Cite EPSG codes when relevant."),
        height=100
    )

    if st.button("🗑️ Clear Chat"):
        st.session_state.messages = []
        st.rerun()

# ─────────────────────────────────────────
# IMAGE UPLOAD
# ─────────────────────────────────────────
col1, col2 = st.columns([1, 1])

with col1:
    uploaded = st.file_uploader(
        "📤 Upload satellite image",
        type=["png", "jpg", "jpeg"]
    )

    if uploaded:
        image = Image.open(uploaded)
        st.image(image, caption="Uploaded image", use_column_width=True)
        st.session_state.current_image = image
    elif "current_image" in st.session_state:
        st.image(st.session_state.current_image,
                 caption="Current image", use_column_width=True)

# ─────────────────────────────────────────
# CHAT INTERFACE
# ─────────────────────────────────────────
with col2:
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # Display chat history
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    # Chat input
    if prompt := st.chat_input("Ask about the image..."):
        if not api_key:
            st.error("⚠️ Please enter your API key in the sidebar")
            st.stop()

        if "current_image" not in st.session_state:
            st.error("⚠️ Please upload an image first")
            st.stop()

        # Configure Gemini
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel(model_choice,
                                       system_instruction=system_prompt)

        # Add user message
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)

        # Stream response
        with st.chat_message("assistant"):
            response = model.generate_content(
                [prompt, st.session_state.current_image],
                stream=True
            )
            full = st.write_stream(chunk.text for chunk in response)

        st.session_state.messages.append({"role": "assistant", "content": full})
'''

# Save the app to disk
with open('satellite_chat.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_app_code)

print("✅ satellite_chat.py created")
print("\nTo run:")
print("  pip install streamlit google-generativeai Pillow")
print("  streamlit run satellite_chat.py")

✅ satellite_chat.py created

To run:
  pip install streamlit google-generativeai Pillow
  streamlit run satellite_chat.py


### 🔍 App walkthrough

**Sidebar:**
- API key input with `type="password"` (won't be shown)
- Model selection (Flash vs Pro)
- Customizable system prompt — the user can tweak the personality

**Main area (column 1):**
- File uploader for the image
- We save the image in `st.session_state` so it persists across reruns

**Main area (column 2):**
- Chat history displayed via `st.chat_message`
- `st.chat_input` for input
- `st.write_stream` renders the streaming response in real time

**The trick:**
We send `[prompt, st.session_state.current_image]` every time — meaning the image is re-sent with each question. This is **not efficient** but is simple.

**Production improvement:** use `model.start_chat()` and put the image in the first message only, then send text-only follow-ups.

### 🎬 Demo time

Run the app and try these questions on a real satellite image:

1. *"What land cover types do you see?"*
2. *"Estimate the urban density on a scale of 1–10"*
3. *"Are there visible signs of recent construction?"*
4. *"What CRS would be appropriate for mapping this area?"*
5. *"Suggest 3 GIS analyses I could perform on this area"*

## 🐍 3.3 — Use Case 3: QGIS Python Script Generator

**Time:** 30 minutes

### The scenario

As a GIS engineer, you find yourself writing QGIS Python scripts every week for repetitive tasks:
- Buffer + intersect
- Reproject layers
- Calculate statistics from raster
- Export attribute tables

Instead of writing them by hand, let the AI write them. **But with care.**

### Promise vs reality

| Promise | Reality |
|---------|---------|
| The AI writes perfect QGIS scripts | Usually 70–80% correct, but there are pitfalls |
| It uses the latest API | It might use deprecated methods |
| It handles edge cases | No — you have to think about those yourself |

**Rule:** the AI is a sharp junior GIS dev. **Trust but verify.**

In [33]:
# ============================================
# 3.3 — QGIS Script Generator
# ============================================

QGIS_EXPERT_PROMPT = """You are an expert in QGIS Python scripting (PyQGIS).

When generating scripts:
1. Use the modern PyQGIS API (qgis.core, qgis.processing) — NOT the legacy QGIS 2 API
2. Use processing.run() for analysis operations whenever possible
3. Always include error handling for missing layers
4. Include a docstring explaining what the script does
5. Add inline comments for non-obvious steps
6. Output ONLY the Python code, no markdown fences, no explanation
7. Assume the script will run in the QGIS Python Console

Common pitfalls to AVOID:
- Don't confuse PyQGIS with ArcPy (they're different!)
- Don't assume layers exist — always check
- Don't hardcode CRS — read from the layer
"""

def generate_qgis_script(task: str) -> str:
    """Generate a PyQGIS script for a given task."""
    model = genai.GenerativeModel(
        'gemini-2.5-flash',
        system_instruction=QGIS_EXPERT_PROMPT
    )

    response = model.generate_content(
        f"Write a PyQGIS script for this task:\n\n{task}",
        generation_config={'temperature': 0.2}
    )
    return response.text

# Test it on a real GIS task
task = """
I have two layers loaded in QGIS:
- 'roads' (line layer)
- 'parcels' (polygon layer)

Buffer the roads by 50 meters, then find all parcels that intersect
with the buffer. Create a new memory layer called 'affected_parcels'
with the result and add it to the project.
"""

script = generate_qgis_script(task)
print(script)

```python
import processing
from qgis.core import QgsProject, QgsVectorLayer, QgsWkbTypes
from qgis.PyQt.QtCore import QVariant

def find_affected_parcels():
    """
    Buffers the 'roads' layer by 50 meters, then finds all 'parcels' that
    intersect with the buffer. Creates a new memory layer called 'affected_parcels'
    with the result and adds it to the project.
    """

    # Define layer names
    roads_layer_name = 'roads'
    parcels_layer_name = 'parcels'

    project = QgsProject.instance()

    # 1. Get input layers from the QGIS project
    roads_layers = project.mapLayersByName(roads_layer_name)
    parcels_layers = project.mapLayersByName(parcels_layer_name)

    # Error handling for missing layers
    if not roads_layers:
        print(f"Error: Layer '{roads_layer_name}' not found in the project.")
        return
    roads_layer = roads_layers[0] # Get the first layer if multiple exist with the same name

    if not parcels_layers:
        print(f"Error: Layer '{parce

### 🔍 Verifying AI output

**This is what distinguishes a senior engineer.** After the AI generates code, check:

#### Checklist for QGIS scripts from the AI

| Check | Why |
|-------|-----|
| Imports present and correct? | The AI sometimes forgets `from qgis.core import ...` |
| Methods are modern? | `iface.legendInterface()` has been deprecated for years |
| CRS handling | If layers are in different CRSes, the buffer will fail |
| Memory layer creation | Syntax changes between versions |
| Error handling | If a layer is missing, the script crashes |

#### Common AI mistakes in PyQGIS

1. **Confusing PyQGIS with ArcPy:**
   ❌ `arcpy.Buffer_analysis(...)`
   ✅ `processing.run('native:buffer', {...})`

2. **Outdated buffer method:**
   ❌ `layer.buffer(50)` (doesn't exist)
   ✅ `processing.run('native:buffer', {'INPUT': layer, 'DISTANCE': 50, ...})`

3. **Wrong CRS assumption:**
   ❌ `buffer_distance = 50` (in a degree-based CRS, 50 degrees = half the planet!)
   ✅ Reproject to a projected CRS first, then buffer

4. **Hallucinated processing algorithms:**
   ❌ `processing.run('native:advanced_buffer', ...)` (doesn't exist)
   ✅ Verify the algorithm name with `processing.algorithmHelp(...)`

### 🛡️ Defensive Vibe Coding

**Correct pattern:** instead of copying code and running it, follow this workflow:

1. The AI generates the code
2. **You** read it (don't run immediately)
3. **You** run it on a **small sample dataset**
4. **You** visually check the output in QGIS
5. **You** run it on the real dataset

**Anti-pattern:**
❌ AI generates → you run immediately on a 10 GB dataset → code corrupts the workspace.

### 🎯 The hidden skill

In the vibe-coding era, the **senior engineer** doesn't write code better than the junior. They:
- **Read** better
- **Test** better
- **Catch errors** faster
- **Understand** why code might break

That's what you'll do in the lab.

## ⚠️ 3.4 — When AI Fails (GIS-Specific Pitfalls)

**Time:** 15 minutes

### AI is not infallible

In the last 75 minutes we've seen the AI do wonderful things. Now let's talk about **when the AI fails badly in GIS specifically**.

---

### Pitfall 1: Coordinate Systems

**Problem:** the LLM defaults to WGS84 / lat-lon (EPSG:4326), even when context suggests otherwise.

**Example:**
*"Buffer this layer by 1000"* — the AI might assume meters, but if the layer is in EPSG:4326, 1000 = 1000 degrees (meaningless).

**Fix:**
Always mention the CRS in your prompt:
*"The layer is in EPSG:32636 (UTM Zone 36N). Buffer by 1000 meters."*

---

### Pitfall 2: Hallucinated EPSG Codes

**A real dangerous example:**
I asked the AI: *"What's the EPSG code for Saudi Arabia's national grid?"*
The AI confidently replied: **EPSG:9999** (which doesn't exist).

**Fix:**
- Verify any EPSG code at [epsg.io](https://epsg.io)
- Cross-check with the PROJ database
- If the AI returns a code, ask it to explain what it means — if hallucinating, the explanation will be inconsistent

---

### Pitfall 3: Outdated Library APIs

**Problem:** the LLM was trained on older data. It may use:
- `geopandas.GeoDataFrame.to_crs(epsg=4326)` instead of the modern `to_crs('EPSG:4326')`
- `shapely` methods that are now deprecated
- ArcGIS Pro API methods from 2019

**Fix:**
- Read the official docs after the AI generates
- Try the code in an isolated environment first
- If you hit an error, give the AI the error message and ask it to fix

---

### Pitfall 4: Geometric Reasoning

**Problem:** the LLM **sees** images but does not **measure** them. When it says:
*"The urban area covers approximately 30% of the image"*
that's a **visual guess**, not an actual calculation.

**Fix:**
For quantitative analysis (areas, distances, counts), use:
- Traditional GIS tools (QGIS, ArcGIS)
- Computer vision libraries (OpenCV, scikit-image)
- Specialized ML models (Mask R-CNN for segmentation)

LLMs are for **narrative and exploration**, not for **measurement**.

---

### Pitfall 5: Spatial Statistics

**Problem:** the LLM confuses similar concepts:
- Moran's I vs Geary's C
- Kriging vs IDW
- Local vs global statistics

**Fix:**
When asking the AI about spatial statistics:
1. Be very specific in the question
2. Ask for step-by-step reasoning
3. Ask for references to the formulas
4. Verify against a textbook or peer-reviewed paper

---

### 🚨 The Cardinal Rule

> **AI = junior GIS analyst.**
> Smart, fast, broad knowledge.
> But it needs supervision from a senior.
> The **senior** is **you**.

### 💡 Mental Model for trust

| Task Type | Trust Level |
|-----------|-------------|
| Code syntax (Python, SQL) | 🟢 90% — usually correct |
| Code logic (algorithms) | 🟡 70% — verify |
| GIS concepts explanation | 🟡 70% — usually right |
| EPSG codes & projections | 🔴 50% — always verify |
| Quantitative measurements from images | 🔴 30% — use traditional tools |
| Real-time data | 🔴 0% — LLMs have no live data |

---

# 🧪 Part 4: Lab Brief — Build a GIS Assistant

**Duration:** 30-minute briefing + the following week for implementation

## 🎯 The Assignment

Build an **AI-powered GIS Assistant** that helps a GIS engineer with their daily work.

Pick whichever specialty appeals to you:

| Specialty | Use case |
|-----------|----------|
| **🐍 QGIS Script Helper** | Generates PyQGIS scripts for repetitive tasks |
| **🛰️ Remote Sensing Analyst** | Analyzes satellite images and answers questions |
| **📍 Address Parser & Geocoder** | Understands messy Egyptian addresses |
| **🗺️ Cartographer Assistant** | Helps with map design and symbology |
| **🌊 Spatial Analyst** | Explains spatial statistics and suggests analyses |

Or **propose your own specialty** (must be GIS-related).

## 📋 Requirements

### Mandatory Features

- [ ] **Streamlit UI** with a sidebar for settings and a main area for chat
- [ ] **API key input** secure (`type="password"`)
- [ ] **Model selection** for at least 2 models (e.g., Gemini Flash and Pro)
- [ ] **Customizable system prompt** with presets for the specialty
- [ ] **Temperature control** slider
- [ ] **Streaming responses**
- [ ] **Chat history** persistent in session
- [ ] **Clear chat button**

### Bonus Features (optional)

- [ ] **Image upload** (if the specialty benefits from it)
- [ ] **Structured output** (JSON mode for extraction tasks)
- [ ] **Export chat to markdown/PDF**
- [ ] **Preset prompts** ("Generate buffer script", "Analyze NDVI", …)
- [ ] **Token counter** in real-time
- [ ] **Multi-provider support** (Gemini + Groq + OpenRouter)
- [ ] **Deploy on Streamlit Cloud** and share the link

## 📦 Deliverables

**Not the code!** The AI will write the code. The real deliverables:

### 1. GitHub Repository containing:
- `app.py` — the Streamlit app
- `requirements.txt` — the dependencies
- `README.md` — setup + screenshots + demo gif
- `.env.example` — template for API keys
- `.gitignore` — excludes `.env`

### 2. Design Document (`DESIGN.md`) — this is where the real grading happens

#### Section A: System Prompt Justification (300 words)
- Which persona did you choose?
- Why did you write the system prompt the way you did?
- What edge cases does it handle?
- Share 2–3 versions you tried and why you settled on the final one

#### Section B: Provider Selection Memo (200 words)
- Did you pick Gemini, Groq, or OpenRouter? Why?
- What are the tradeoffs (speed vs quality vs cost)?
- If 100 users hit the app concurrently, will the setup scale?

#### Section C: Test Cases (minimum of 10)
Have the AI answer 10 questions and document:
- 5 happy path (natural questions for the specialty)
- 3 edge cases (out of scope question, ambiguous, Arabic-language)
- 2 adversarial (trying to make the AI do something wrong)

For every case:
- The question
- The actual response
- Reflection: is the response useful? What could be improved?

#### Section D: Limitations & Failures (200 words)
- What is the app **unable** to do?
- What's the biggest mistake you saw the AI make?
- What makes this app dangerous if used without understanding?

## 📊 Grading Rubric (100 points)

| Section | Weight | Criteria |
|---------|--------|----------|
| **System Prompt Quality** | 25% | Reflects real domain expertise? Handles edge cases? |
| **Provider Selection Reasoning** | 15% | Is the choice justified? Are tradeoffs understood? |
| **Test Cases Thoughtfulness** | 25% | Are the cases revealing? Is the reflection deep? |
| **Limitations Awareness** | 15% | Does the student know the app's limits? |
| **Code Quality** | 10% | Does it run? Is it clean? Documented? |
| **UX & Polish** | 10% | Clear UI? Error handling? |

**Notice:** code is only 10%. The real value is in the thinking and reflection.

## 🎬 Demo: Starting Point

Use this as a starter and build on it.

In [ ]:
# ============================================
# LAB STARTER TEMPLATE
# Modify and extend for your specialty
# ============================================

starter_code = '''import streamlit as st
import google.generativeai as genai
import os

# ─────────────────────────────────────────
# TODO: Customize these for your specialty
# ─────────────────────────────────────────
APP_TITLE = "🌍 Your GIS Assistant Name"
APP_ICON = "🌍"

SYSTEM_PROMPTS = {
    "General GIS": "You are a helpful GIS assistant.",
    "QGIS Expert": (
        "You are a PyQGIS expert. Generate modern QGIS Python scripts. "
        "Always use processing.run() for analyses."
    ),
    "Remote Sensing": (
        "You are a remote sensing analyst specializing in satellite imagery "
        "analysis. Cite EPSG codes when relevant."
    ),
    # TODO: Add your specialty preset(s)
}

MODELS = {
    "Gemini 2.5 Flash (Fast)": "gemini-2.5-flash",
    "Gemini 2.5 Pro (Smart)": "gemini-2.5-pro",
    # TODO: Add more if using multi-provider
}

# ─────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────
st.set_page_config(page_title=APP_TITLE, page_icon=APP_ICON)
st.title(APP_TITLE)

# ─────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Settings")

    api_key = st.text_input(
        "Gemini API Key",
        type="password",
        value=os.environ.get("GOOGLE_API_KEY", "")
    )

    model_label = st.selectbox("Model", list(MODELS.keys()))
    model_id = MODELS[model_label]

    preset = st.selectbox("Specialty", list(SYSTEM_PROMPTS.keys()))
    system_prompt = st.text_area(
        "System Prompt",
        value=SYSTEM_PROMPTS[preset],
        height=120
    )

    temperature = st.slider("Temperature", 0.0, 2.0, 0.5, 0.1)

    if st.button("🗑️ Clear Chat"):
        st.session_state.messages = []
        st.rerun()

# ─────────────────────────────────────────
# CHAT INTERFACE
# TODO: Complete this!
# Hints:
#   1. Initialize st.session_state.messages = []
#   2. Display chat history with st.chat_message
#   3. Accept input with st.chat_input
#   4. Configure genai with api_key
#   5. Create model with system_instruction=system_prompt
#   6. Stream response with st.write_stream
#   7. Append to history
# ─────────────────────────────────────────

st.info("👆 Complete the chat interface using vibe coding!")
'''

with open('gis_assistant_starter.py', 'w', encoding='utf-8') as f:
    f.write(starter_code)

print("✅ gis_assistant_starter.py created")
print("\nNext steps:")
print("  1. Open the file in Cursor/Claude Code")
print("  2. Vibe-code the chat interface (the TODO section)")
print("  3. Customize SYSTEM_PROMPTS for your specialty")
print("  4. Test thoroughly — document test cases")
print("  5. Deploy to Streamlit Cloud")
print("  6. Write DESIGN.md")

---

# 📎 Appendices

## Appendix A — Answers to Part 1.4 Q&A

**Q1: 1000 satellite tiles — Chat UI or API?**
**A:** API. The Chat UI = manual = impossible at 1000 tiles. The API = loop in code = minutes.

**Q2: System prompt says "You are a Python expert" and user asks about GIS?**
**A:** The AI **will answer**. The system prompt guides behavior but doesn't "lock" the AI to one topic. If you want strict scope, the system prompt must contain an explicit refusal rule: *"If asked about anything other than Python, politely decline and redirect."*

**Q3: temperature=0.0, same answer twice?**
**A:** **Theoretically yes, in practice usually.** Temperature 0 suppresses randomness but there are other sources of non-determinism (e.g., GPU batching). For absolute determinism, use a `seed` parameter if the API supports it.

**Q4: Why is streaming not useful for background scripts?**
**A:** Streaming adds complexity (looping over chunks) and brings no UX benefit in a background job. Also, for structured output (JSON), you need the full response to parse.

**Q5: Analyze satellite image — Groq Llama 70B or Gemini Flash?**
**A:** **Gemini Flash.** Groq Llama 3.3 70B = text only. You must use a vision-enabled model. Gemini Flash is free with strong vision.

## Appendix B — Cheat Sheet

### Provider Setup

```python
# Gemini
import google.generativeai as genai
genai.configure(api_key='...')
model = genai.GenerativeModel('gemini-2.5-flash', system_instruction='...')
response = model.generate_content('hello', stream=True)

# Groq
from groq import Groq
client = Groq(api_key='...')
response = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': 'hello'}],
    stream=True,
)

# OpenRouter (uses OpenAI SDK)
from openai import OpenAI
client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key='...')
response = client.chat.completions.create(
    model='meta-llama/llama-3.3-70b-instruct:free',
    messages=[{'role': 'user', 'content': 'hello'}],
)
```

### Parameter Quick Reference

| Parameter | Range | When to change it |
|-----------|-------|-------------------|
| `temperature` | 0.0 – 2.0 | 0.1 for code, 0.7 for chat, 0.9 for creative |
| `max_tokens` | 1 – context limit | 50 for classification, 2000 for reports |
| `stream` | True / False | True for UIs, False for scripts |
| `top_p` | 0.0 – 1.0 | Usually leave at 1.0 |

### "Aha" Patterns

1. **OpenAI-compatible format:** Groq + OpenRouter + most providers = nearly identical code
2. **Gemini = different SDK but same concepts:** `system_instruction`, `generate_content`, `stream=True`
3. **Multimodal = list of [text, image, text, image, ...]** — Gemini handles the rest
4. **JSON mode:** `response_mime_type='application/json'` (Gemini) or `response_format={'type': 'json_object'}` (OpenAI)

## Appendix C — Common Errors & Fixes

| Error | Meaning | Fix |
|-------|---------|-----|
| `429 Too Many Requests` | Rate limit | Wait + retry with exponential backoff |
| `401 Unauthorized` | Wrong API key | Regenerate the key, check env var |
| `400 Bad Request` | Bad request format | Read the error message — usually a missing field |
| `503 Service Unavailable` | Provider down | Switch to fallback provider |
| `Empty response` | Safety filter rejected | Reword the prompt |
| `Truncated response` | `max_tokens` too small | Increase max_tokens |
| `ModuleNotFoundError` | SDK not installed | `pip install groq google-generativeai openai` |

### Production-grade retry pattern

```python
import time
from typing import Callable, Any

def with_retries(func: Callable, max_retries: int = 3) -> Any:
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt   # 1, 2, 4 seconds
            print(f"Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
```

## Appendix D — Resources

### 📖 Official Docs

| Resource | URL |
|----------|-----|
| Google AI Studio | https://aistudio.google.com |
| Gemini API docs | https://ai.google.dev/docs |
| Groq Console | https://console.groq.com |
| OpenRouter | https://openrouter.ai/docs |
| Streamlit Docs | https://docs.streamlit.io |
| Ollama (local models) | https://ollama.com |

### 🌍 GIS-specific

| Resource | URL |
|----------|-----|
| EPSG Registry | https://epsg.io |
| PyQGIS Cookbook | https://docs.qgis.org/latest/en/docs/pyqgis_developer_cookbook/ |
| GeoPandas Docs | https://geopandas.org |
| Rasterio Docs | https://rasterio.readthedocs.io |
| USGS EarthExplorer (free satellite) | https://earthexplorer.usgs.gov |
| Copernicus Open Access Hub | https://scihub.copernicus.eu |

### 🎓 Self-Study Courses (for the 6 self-study hours)

| Course | Duration |
|--------|----------|
| [AI Python for Beginners (DeepLearning.AI)](https://www.deeplearning.ai/short-courses/ai-python-for-beginners/) | 2h |
| [Building Systems with the ChatGPT API](https://www.deeplearning.ai/short-courses/building-systems-with-chatgpt/) | 1h |
| [ChatGPT Prompt Engineering for Devs](https://www.deeplearning.ai/short-courses/chatgpt-prompt-engineering-for-developers/) | 1h |
| [Streamlit Crash Course (official)](https://docs.streamlit.io/get-started) | 1h |
| Practice: rebuild today's demos from scratch | 1h |

## Appendix E — Self-Reflection Questions

Before closing the day, think about these:

1. **Vibe coding vs traditional coding:**
   What did you learn today about vibe coding that's different from yesterday's lecture?
   *(Hint: yesterday was "how to write". Today was "what to ask for".)*

2. **Reading code:**
   Remember the moment in Part 2.1 when the AI wrote code and you read it with understanding? Is this a different feeling from writing the code yourself?

3. **Provider mental model:**
   If you have 5 different GIS projects, each needing a different provider, can you map every project to a provider quickly?

4. **AI failure modes:**
   What scared you the most about AI in a GIS context? How would you mitigate it?

5. **The lab:**
   If you do the lab honestly (not by "copy-pasting assistant code"), you'll come out with skills that last years. Reflection in `DESIGN.md` = the real learning.

---

## ✅ Day 3 Checklist

- [ ] I understand the difference between Chat UI and API
- [ ] I know the 5 components of any LLM call (messages, model, temperature, max_tokens, stream)
- [ ] I know when to use Gemini vs Groq vs OpenRouter
- [ ] I have API keys for at least 3 providers
- [ ] I tried the first API call via vibe coding
- [ ] I tried multimodal (image analysis)
- [ ] I tried the 3.1 address parser
- [ ] I understand when AI fails in GIS
- [ ] I read the lab brief and chose a specialty
- [ ] I started thinking about the system prompt for the lab

---

## 🚀 Day 4 Preview: RAG Systems

Tomorrow we'll cover:
- **RAG (Retrieval-Augmented Generation):** how to make the LLM answer from your own documents
- **Vector databases** (ChromaDB)
- **Embeddings** for spatial data
- **Use case:** "Chat with your shapefile" — asking about an attribute table in natural language

**Homework for Day 4:**
1. Deploy your lab assignment (if possible)
2. Watch [LangChain Chat with Your Data](https://www.deeplearning.ai/short-courses/langchain-chat-with-your-data/) (1 hour)
3. Read: [What are embeddings?](https://platform.openai.com/docs/guides/embeddings)

---

## 🧹 Cleanup

Run the next cell to clear API keys from memory.

In [ ]:
# Clear API keys from memory (security best practice)
clear_api_keys()

---

*Day 3 — AI-Powered GIS · ITI Gen AI Course · GIS Track*
*Designed for students who've completed Vibe Coding (Day 2)*
*Last updated: May 2026*